<a href="https://colab.research.google.com/github/abhisekchaudhuri/PythonLabs/blob/AI-Explorations/The_AI_Council.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from openai import OpenAI
from anthropic import Anthropic
import google.generativeai as genai
from google.colab import userdata # Import userdata for secure key storage

# --- 1. API Key Setup ---
# It's recommended to store API keys using Colab's secret manager for security.

# OpenAI (ChatGPT)
openai_api_key = userdata.get("OPENAI_API_KEY") # Retrieve from Colab secrets
openai_client = None
if openai_api_key:
    openai_client = OpenAI(api_key=openai_api_key)
else:
    print("OPENAI_API_KEY not found in Colab secrets. ChatGPT functionality will be limited.")

# Anthropic (Claude)
anthropic_api_key = userdata.get("ANTHROPIC_API_KEY") # Retrieve from Colab secrets
anthropic_client = None
if anthropic_api_key:
    anthropic_client = Anthropic(api_key=anthropic_api_key)
else:
    print("ANTHROPIC_API_KEY not found in Colab secrets. Claude functionality will be limited.")

# Google Gemini
gemini_api_key = userdata.get("GEMINI_API_KEY") # Retrieve from Colab secrets
gemini_model = None
if gemini_api_key:
    genai.configure(api_key=gemini_api_key)
    gemini_model = genai.GenerativeModel('gemini-1.5-flash') # Using a capable and efficient Gemini model
else:
    print("GEMINI_API_KEY not found in Colab secrets. Gemini functionality will be limited.")

print("API clients initialized.")

# --- 2. Interface Logic ---
def creative_llm_interface(query: str, num_loops: int = 2):
    """
    Orchestrates LLM interaction: Claude builds, Gemini critiques, ChatGPT decides and analyzes.
    This process runs for a specified number of loops.
    """
    print(f"\nProcessing query: {query}")

    # Track the entire process for final analysis
    process_history = []
    current_answer = ""
    claude_initial_answer = "" # Store the very first answer from Claude

    for i in range(num_loops):
        print(f"\n--- Loop {i+1}/{num_loops} ---")
        loop_results = {"loop_number": i + 1}

        # Step 1: Claude for building the initial answer or refining
        print(f"--- Claude {'Building Initial Answer' if i == 0 else 'Refining Answer'} ---")
        claude_response_text = ""
        claude_prompt = ""

        if i == 0:
            claude_prompt = query
            if anthropic_client:
                try:
                    print("Sending initial query to Claude...")
                    claude_response = anthropic_client.messages.create(
                        model="claude-3-5-sonnet-200k", # Using a powerful Claude model
                        max_tokens=2048,
                        messages=[{"role": "user", "content": claude_prompt}]
                    )
                    claude_response_text = claude_response.content[0].text
                    claude_initial_answer = claude_response_text # Save the initial answer
                    print("Claude's initial answer received.")
                except Exception as e:
                    claude_response_text = f"Error with Claude initial generation: {e}"
                    print(claude_response_text)
            else:
                claude_response_text = "Claude API key not provided or client not initialized."
                print(claude_response_text)
            current_answer = claude_response_text
            loop_results["claude_initial_answer"] = claude_response_text
        else:
            # If not the first loop, Claude refines based on ChatGPT's guidance from previous loop
            if anthropic_client and process_history[-1].get("chatgpt_guidance_for_claude"):
                try:
                    print("Sending refinement prompt to Claude...")
                    claude_response = anthropic_client.messages.create(
                        model="claude-3-5-sonnet-200k",
                        max_tokens=2048,
                        messages=[{"role": "user", "content": process_history[-1]["chatgpt_guidance_for_claude"]}]
                    )
                    claude_response_text = claude_response.content[0].text
                    print("Claude's refined answer received.")
                except Exception as e:
                    claude_response_text = f"Error with Claude refinement: {e}"
                    print(claude_response_text)
            else:
                claude_response_text = "Claude API key not provided, client not initialized, or no ChatGPT guidance for refinement."
                print(claude_response_text)
            current_answer = claude_response_text
            loop_results["claude_refined_answer"] = claude_response_text

        print("Current answer summary:")
        print(current_answer[:500] + "..." if len(current_answer) > 500 else current_answer)


        # Step 2: Gemini to critique the current answer
        print("\n--- Gemini Critiquing ---")
        gemini_critique = ""
        if gemini_model:
            try:
                print("Sending critique request to Gemini...")
                gemini_prompt = f"Critique the following answer based on the original query: '{query}'\n\nAnswer to critique: '{current_answer}'\n\nProvide constructive feedback, identify weaknesses, and suggest areas for improvement. Be concise and actionable."
                gemini_response = gemini_model.generate_content(gemini_prompt)
                gemini_critique = gemini_response.text
                print("Gemini's critique received.")
            except Exception as e:
                gemini_critique = f"Error with Gemini critique: {e}"
                print(gemini_critique)
        else:
            gemini_critique = "Gemini API key not provided or model not initialized."
            print(gemini_critique)

        loop_results["gemini_critique"] = gemini_critique
        print("Gemini's critique summary:")
        print(gemini_critique[:500] + "..." if len(gemini_critique) > 500 else gemini_critique)


        # Step 3: ChatGPT for decision making and formulating Claude's next prompt
        print("\n--- ChatGPT Decision Making & Refinement Guidance ---")
        chatgpt_guidance_for_claude = ""
        chatgpt_analysis_of_round = ""
        if openai_client:
            try:
                print("Sending request to ChatGPT for guidance...")
                chatgpt_prompt = f"Original Query: '{query}'\n\nCurrent Answer (from Claude): '{current_answer}'\n\nGemini's Critique: '{gemini_critique}'\n\nBased on the original query, the current answer, and Gemini's critique, please do two things:\n1. Provide a concise analysis of the current answer's strengths and weaknesses, considering Gemini's critique and your own insights.\n2. Formulate a detailed and clear prompt for Claude to refine the 'Current Answer'. This prompt should incorporate Gemini's feedback, your analysis, and any additional logic you deem necessary to improve the answer. Ensure the prompt is actionable for Claude.\n\nStructure your response with two clear sections: 'ChatGPT Analysis:' and 'Prompt for Claude:'"
                chatgpt_response = openai_client.chat.completions.create(
                    model="gpt-4o-mini", # Using a capable OpenAI model
                    messages=[{"role": "user", "content": chatgpt_prompt}],
                    max_tokens=2048
                )
                chatgpt_full_response = chatgpt_response.choices[0].message.content
                print("ChatGPT's guidance received.")

                # Attempt to parse ChatGPT's response
                if "ChatGPT Analysis:" in chatgpt_full_response and "Prompt for Claude:" in chatgpt_full_response:
                    analysis_start = chatgpt_full_response.find("ChatGPT Analysis:") + len("ChatGPT Analysis:")
                    prompt_start = chatgpt_full_response.find("Prompt for Claude:")
                    chatgpt_analysis_of_round = chatgpt_full_response[analysis_start:prompt_start].strip()
                    chatgpt_guidance_for_claude = chatgpt_full_response[prompt_start + len("Prompt for Claude:"):].strip()
                else:
                    chatgpt_analysis_of_round = "Could not parse ChatGPT's structured response." # Fallback analysis
                    chatgpt_guidance_for_claude = f"Based on original query: '{query}', current answer: '{current_answer}', and Gemini's critique: '{gemini_critique}', refine the answer to be more comprehensive and address weaknesses." # Fallback prompt
                    print("Warning: ChatGPT's response structure not as expected. Using fallback prompt for Claude.")

            except Exception as e:
                chatgpt_analysis_of_round = f"Error with ChatGPT analysis and guidance: {e}"
                chatgpt_guidance_for_claude = f"Based on original query: '{query}', current answer: '{current_answer}', and Gemini's critique: '{gemini_critique}', refine the answer to be more comprehensive and address weaknesses." # Fallback prompt
                print(chatgpt_analysis_of_round)
        else:
            chatgpt_analysis_of_round = "ChatGPT API key not provided or client not initialized."
            chatgpt_guidance_for_claude = f"Based on original query: '{query}', current answer: '{current_answer}', and Gemini's critique: '{gemini_critique}', refine the answer to be more comprehensive and address weaknesses." # Fallback prompt
            print(chatgpt_analysis_of_round)

        loop_results["chatgpt_analysis_of_round"] = chatgpt_analysis_of_round
        loop_results["chatgpt_guidance_for_claude"] = chatgpt_guidance_for_claude
        print("ChatGPT's analysis for this round summary:")
        print(chatgpt_analysis_of_round[:500] + "..." if len(chatgpt_analysis_of_round) > 500 else chatgpt_analysis_of_round)
        print("ChatGPT's prompt for Claude's next iteration summary:")
        print(chatgpt_guidance_for_claude[:500] + "..." if len(chatgpt_guidance_for_claude) > 500 else chatgpt_guidance_for_claude)

        process_history.append(loop_results)

    # Final Answer is the last refined answer from Claude
    final_answer = current_answer

    # Final ChatGPT Analysis of the entire process
    print("\n--- Final ChatGPT Comparative Analysis ---")
    chatgpt_final_analysis = ""
    if openai_client:
        try:
            print("Requesting final comparative analysis from ChatGPT...")
            final_analysis_prompt = f"Original Query: '{query}'\n\n"
            final_analysis_prompt += f"Claude's Initial Answer: '{claude_initial_answer}'\n\n"

            for idx, res in enumerate(process_history):
                if "claude_refined_answer" in res:
                    final_analysis_prompt += f"Claude's Answer (Loop {idx+1}): '{res['claude_refined_answer']}'\n\n"
                else: # First loop uses initial answer
                    final_analysis_prompt += f"Claude's Answer (Loop {idx+1}): '{res['claude_initial_answer']}'\n\n"
                final_analysis_prompt += f"Gemini's Critique (Loop {idx+1}): '{res['gemini_critique']}'\n\n"
                final_analysis_prompt += f"ChatGPT's Analysis & Guidance (Loop {idx+1}): '{res['chatgpt_analysis_of_round']}'\n\n"
            final_analysis_prompt += f"Final Answer (from Claude after {num_loops} loops): '{final_answer}'\n\n"

            final_analysis_prompt += "Based on the entire interaction history (original query, Claude's initial and refined answers, Gemini's critiques, and your own guidance at each step), provide a comprehensive comparative analysis. Highlight the following:\n1. Strengths and weaknesses of each LLM's contribution (Claude's generation, Gemini's critique, ChatGPT's guidance).\n2. How the answer evolved over the loops.\n3. Key differences and improvements made.\n4. Any remaining gaps or areas for further enhancement.\n5. Your final assessment of the solution and the multi-LLM process. Be detailed and insightful."

            chatgpt_response = openai_client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": final_analysis_prompt}],
                max_tokens=4096 # Allow more tokens for comprehensive analysis
            )
            chatgpt_final_analysis = chatgpt_response.choices[0].message.content
            print("ChatGPT's final analysis received.")
        except Exception as e:
            chatgpt_final_analysis = f"Error with ChatGPT final analysis: {e}"
            print(chatgpt_final_analysis)
    else:
        chatgpt_final_analysis = "ChatGPT API key not provided or client not initialized, skipping final analysis."
        print(chatgpt_final_analysis)


    print("\n--- Final Consolidated Answer ---")
    print(final_answer)

    print("\n--- ChatGPT's Final Process Analysis ---")
    print(chatgpt_final_analysis)

    return {
        "final_answer": final_answer,
        "chatgpt_final_analysis": chatgpt_final_analysis,
        "process_history": process_history
    }

ModuleNotFoundError: No module named 'anthropic'

## User Input and Processing

This section allows you to provide your query, either by uploading a file (like a resume) or by directly typing text.

In [3]:
# Install PyPDF2 for PDF processing if not already installed
!pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.9 MB/s eta 0:00:00


In [2]:
# Install libraries for Word and Excel processing
!pip install python-docx openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 4.1 MB/s eta 0:00:00


In [3]:
from google.colab import files
import PyPDF2
import io
from docx import Document # Import for .docx files
import openpyxl # Import for .xlsx files

def process_uploaded_file():
    print("Please upload your input file (e.g., .txt, .pdf, .docx, .xlsx). Optional.")
    uploaded = files.upload()

    if not uploaded:
        print("No file uploaded. You can proceed with a direct text query.")
        return ""

    file_content = ""
    for filename, content in uploaded.items():
        print(f'Processing file: "{filename}" ({len(content)} bytes)')

        if filename.lower().endswith('.pdf'):
            try:
                pdf_file = io.BytesIO(content)
                pdf_reader = PyPDF2.PdfReader(pdf_file)
                text = ""
                for page_num in range(len(pdf_reader.pages)):
                    text += pdf_reader.pages[page_num].extract_text() or ""
                file_content = text
                print("Successfully extracted text from PDF.")
            except Exception as e:
                print(f"Error processing PDF file: {e}. Please ensure it's a valid PDF.")
                file_content = ""
        elif filename.lower().endswith(('.docx')):
            try:
                doc = Document(io.BytesIO(content))
                text = []
                for paragraph in doc.paragraphs:
                    text.append(paragraph.text)
                file_content = '\n'.join(text)
                print("Successfully extracted text from DOCX.")
            except Exception as e:
                print(f"Error processing DOCX file: {e}. Please ensure it's a valid DOCX.")
                file_content = ""
        elif filename.lower().endswith(('.xlsx')):
            try:
                workbook = openpyxl.load_workbook(io.BytesIO(content))
                text = []
                for sheet_name in workbook.sheetnames:
                    sheet = workbook[sheet_name]
                    for row in sheet.iter_rows():
                        row_values = [str(cell.value) if cell.value is not None else "" for cell in row]
                        text.append('\t'.join(row_values))
                file_content = '\n'.join(text)
                print("Successfully extracted text from XLSX.")
            except Exception as e:
                print(f"Error processing XLSX file: {e}. Please ensure it's a valid XLSX.")
                file_content = ""
        elif filename.lower().endswith(('.jpg', '.jpeg', '.png', '.gif')):
            print(f"Image file '{filename}' detected. Automatic text extraction from images requires OCR, which is not implemented in this utility. Please manually provide text or use a text-based document.")
            file_content = ""
        else: # Assume plain text for other files
            try:
                file_content = content.decode('utf-8')
                print("Successfully extracted text from plain text file.")
            except UnicodeDecodeError:
                print(f"Error: Could not decode {filename} as UTF-8. Please upload a plain text file or a PDF.")
            except Exception as e:
                print(f"Error reading file {filename}: {e}")
        break # Process only the first uploaded file

    return file_content

# Call the function to handle file upload and content extraction
uploaded_file_content = process_uploaded_file()

Please upload your input file (e.g., .txt, .pdf, .docx, .xlsx). Optional.


KeyboardInterrupt: 

### Provide Your Query

YouYou can either use the content from the uploaded file or provide a direct text query. If both are provided, the direct text query will take precedence.

After defining your query, run the cell below to initiate the multi-LLM process. You can modify the `main_query` variable for follow-up questions or new tasks.

In [1]:
# --- Define Your Main Query Here ---

# Option 1: Direct text query
# Uncomment the line below and replace with your desired query.
# main_query = "Explain the concept of quantum entanglement in simple terms."

# Option 2: Use content from the uploaded file (if any)
# This will be used if 'main_query' is not explicitly set above, and 'uploaded_file_content' is not empty.
# If you uploaded a resume, you might frame a question around it.
# Example: if uploaded_file_content: main_query = f"Analyze this resume: {uploaded_file_content}"

# Fallback or combined approach
main_query = "" # Initialize to empty string

if 'uploaded_file_content' in locals() and uploaded_file_content: # Check if uploaded_file_content exists and is not empty
    # Example of using uploaded content, adjust as needed for your use case.
    # For a resume, you might want to ask a question *about* the resume.
    main_query = f"Analyze the following document for key skills and experience: {uploaded_file_content}"
    print("Using content from the uploaded file for the main query.")
else:
    # If no file content or you want to override it with a direct text query:
    main_query = "Explain the concept of quantum entanglement in simple terms."
    print("Using direct text query as no file content was provided or it was overridden.")

# --- Run the Multi-LLM Interface ---
if main_query:
    print(f"Initiating multi-LLM process with query: {main_query[:100]}...")
    results = creative_llm_interface(main_query, num_loops=2)

    print("\n--- Final Results Summary ---")
    print("Final Answer:")
    print(results['final_answer'][:500] + "...")
    print("\nChatGPT's Final Analysis:")
    print(results['chatgpt_final_analysis'][:500] + "...")
else:
    print("No query was provided. Please define 'main_query' to proceed.")


Using direct text query as no file content was provided or it was overridden.
Initiating multi-LLM process with query: Explain the concept of quantum entanglement in simple terms....


NameError: name 'creative_llm_interface' is not defined